# MRI image classifier

A self-contained notebook for training a ResNet50 classifier from explicitly labelled 2D images or labelled NIfTI volumes. Select **Runtime → Change runtime type → GPU** when available.

> BraTS 2020 contains tumour cases only and cannot provide tumour-versus-healthy labels by itself. Use reviewed labels. This research demo is not a clinical diagnostic tool. NIfTI mode uses one selected axial slice per volume.


In [ ]:
# Install notebook dependencies. NIfTI support is optional.
%pip -q install tensorflow scikit-learn matplotlib opencv-python-headless nibabel
from pathlib import Path
import csv, json, math
from collections import Counter
from typing import Iterable
import cv2
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
try:
    import nibabel as nib
except ImportError:
    nib = None
print('TensorFlow:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))


## Model and data helpers
Run this cell once before configuring a training run.


In [ ]:
from pathlib import Path
import csv
import json
import math
from collections import Counter
from typing import Iterable

IMAGE_EXTENSIONS = {".bmp", ".jpeg", ".jpg", ".png"}
NIFTI_EXTENSIONS = (".nii", ".nii.gz")


def is_nifti_file(path: Path) -> bool:
    return path.name.lower().endswith(NIFTI_EXTENSIONS)


def collect_image_samples(dataset_path: Path) -> tuple[list[Path], list[str]]:
    """Read images from immediate class folders and require every class to have data."""
    classes = sorted(item.name for item in dataset_path.iterdir() if item.is_dir())
    if len(classes) < 2:
        raise ValueError("Image mode needs at least two class subdirectories.")
    paths, labels = [], []
    empty_classes = []
    for class_name in classes:
        files = sorted(
            item for item in (dataset_path / class_name).rglob("*")
            if item.is_file() and item.suffix.lower() in IMAGE_EXTENSIONS
        )
        if not files:
            empty_classes.append(class_name)
        paths.extend(files)
        labels.extend([class_name] * len(files))
    if empty_classes:
        raise ValueError(f"No supported images in class folders: {', '.join(empty_classes)}")
    return paths, labels


def collect_nifti_samples(dataset_path: Path, manifest_path: Path) -> tuple[list[Path], list[str]]:
    """Read NIfTI paths and reviewed labels from a path,label CSV manifest."""
    paths, labels = [], []
    with manifest_path.open(newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        if not reader.fieldnames or not {"path", "label"}.issubset(reader.fieldnames):
            raise ValueError("Label manifest must have 'path' and 'label' columns.")
        for row_number, row in enumerate(reader, start=2):
            raw_path = (row.get("path") or "").strip()
            label = (row.get("label") or "").strip()
            resolved = Path(raw_path)
            resolved = resolved if resolved.is_absolute() else dataset_path / resolved
            if not raw_path or not label:
                raise ValueError(f"Empty path or label in manifest row {row_number}.")
            if not resolved.is_file() or not is_nifti_file(resolved):
                raise FileNotFoundError(f"Invalid NIfTI path in manifest row {row_number}: {resolved}")
            paths.append(resolved)
            labels.append(label)
    return paths, labels


def validate_samples(paths: Iterable[Path], labels: list[str], validation_size: float) -> list[str]:
    paths = list(paths)
    if len(paths) != len(labels) or not paths:
        raise ValueError("Each usable sample must have exactly one label.")
    counts = Counter(labels)
    if len(counts) < 2:
        raise ValueError("Classification needs at least two non-empty classes.")
    too_small = [label for label, count in counts.items() if count < 2]
    if too_small:
        raise ValueError(f"Each class needs at least two samples; too small: {too_small}")
    validation_count = math.ceil(len(labels) * validation_size)
    if validation_count < len(counts) or len(labels) - validation_count < len(counts):
        raise ValueError("The validation split is too small to contain every class in both splits.")
    return sorted(counts)


def load_nifti_slice(path: Path, image_size: int, nib, cv2, np):
    """Choose the axial slice with most foreground, then robustly scale it to [0, 1]."""
    volume = np.asarray(nib.load(str(path)).get_fdata(dtype=np.float32))
    if volume.ndim != 3 or not np.isfinite(volume).all():
        raise ValueError(f"Expected a finite 3D NIfTI volume, got {volume.shape}: {path}")
    foreground = np.abs(volume) > np.percentile(np.abs(volume), 20)
    image = volume[:, :, int(np.argmax(foreground.sum(axis=(0, 1))))]
    low, high = np.percentile(image, (1, 99))
    if not np.isfinite(low) or high - low < 1e-6:
        raise ValueError(f"NIfTI volume is empty or nearly constant: {path}")
    image = np.clip((image - low) / (high - low), 0.0, 1.0)
    image = cv2.resize(image, (image_size, image_size), interpolation=cv2.INTER_AREA)
    return np.repeat(image[..., np.newaxis], 3, axis=-1).astype(np.float32)


def load_image(path: Path, image_size: int, tf):
    if not path.is_file():
        raise FileNotFoundError(f"Image does not exist: {path}")
    image = tf.keras.utils.load_img(path, target_size=(image_size, image_size), color_mode="rgb")
    return tf.keras.utils.img_to_array(image).astype("float32") / 255.0


def build_model(image_size: int, class_count: int, weights: str, tf, learning_rate: float):
    base = tf.keras.applications.ResNet50(
        include_top=False, weights=None if weights == "none" else weights,
        input_shape=(image_size, image_size, 3),
    )
    base.trainable = False
    inputs = tf.keras.Input(shape=(image_size, image_size, 3))
    x = tf.keras.applications.resnet50.preprocess_input(inputs * 255.0)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    outputs = tf.keras.layers.Dense(class_count, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def unfreeze_top_layers(model, count: int) -> None:
    if count <= 0:
        return
    base = next((layer for layer in model.layers if layer.name.startswith("resnet50")), None)
    if base is None or count > len(base.layers):
        raise ValueError("FINE_TUNE_LAYERS exceeds the ResNet feature extractor size.")
    base.trainable = True
    for layer in base.layers[:-count]:
        layer.trainable = False
    for layer in base.layers[-count:]:
        layer.trainable = layer.__class__.__name__ != "BatchNormalization"


def read_patient_ids(manifest_path: Path, paths: list[Path], dataset_path: Path) -> list[str]:
    """Map every sample to a patient ID from a path,patient_id CSV manifest."""
    mapping = {}
    with manifest_path.open(newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        if not reader.fieldnames or not {"path", "patient_id"}.issubset(reader.fieldnames):
            raise ValueError("Patient manifest must have 'path' and 'patient_id' columns.")
        for row_number, row in enumerate(reader, start=2):
            raw_path, patient_id = (row.get("path") or "").strip(), (row.get("patient_id") or "").strip()
            if not raw_path or not patient_id:
                raise ValueError(f"Empty path or patient_id in patient manifest row {row_number}.")
            candidate = Path(raw_path)
            resolved = candidate if candidate.is_absolute() else dataset_path / candidate
            mapping[str(resolved.resolve())] = patient_id
    missing = [str(item) for item in paths if str(item.resolve()) not in mapping]
    if missing:
        raise ValueError(f"Patient manifest is missing {len(missing)} sample paths; first: {missing[0]}")
    return [mapping[str(item.resolve())] for item in paths]


## Configure data and training

For 2D images, put verified classes in separate folders, such as `data/no_tumor/` and `data/tumor/`. For NIfTI, provide a CSV with `path,label` columns. Relative paths are resolved under `DATASET_PATH`. For multiple samples per patient, provide `patient_ids.csv` with `path,patient_id` columns to keep patients together across the split. Every class needs sufficient independent examples in both splits.


In [ ]:
# Optional: mount Google Drive for persistent data and results.
from google.colab import drive
drive.mount('/content/drive')

DATA_MODE = "image"  # "image" or "nifti"
DATASET_PATH = Path("/content/drive/MyDrive/mri-data")
LABELS_CSV = DATASET_PATH / "labels.csv"  # used only in NIfTI mode
PATIENT_CSV = None  # e.g. DATASET_PATH / "patient_ids.csv"; strongly recommended for multiple samples per patient
OUTPUT_DIR = Path("/content/drive/MyDrive/mri-artifacts")
IMAGE_SIZE, BATCH_SIZE, EPOCHS = 224, 8, 12
VALIDATION_SIZE, SEED = 0.2, 42
WEIGHTS = "imagenet"  # "imagenet" or "none"
LEARNING_RATE = 1e-4
FINE_TUNE_LAYERS = 0  # set a small positive value after a baseline run
FINE_TUNE_LEARNING_RATE = 1e-5


In [ ]:
def train_classifier():
    """Train, evaluate, and save the best validation model in OUTPUT_DIR."""
    if DATA_MODE not in ("image", "nifti"):
        raise ValueError("DATA_MODE must be 'image' or 'nifti'.")
    if WEIGHTS not in ("imagenet", "none"):
        raise ValueError("WEIGHTS must be 'imagenet' or 'none'.")
    if not DATASET_PATH.is_dir():
        raise FileNotFoundError(f"Dataset directory does not exist: {DATASET_PATH}")
    if DATA_MODE == "nifti" and (nib is None or not LABELS_CSV.is_file()):
        raise ValueError("NIfTI mode requires nibabel and an existing labels.csv file.")
    if PATIENT_CSV is not None and not Path(PATIENT_CSV).is_file():
        raise FileNotFoundError(f"Patient manifest does not exist: {PATIENT_CSV}")
    if not 0 < VALIDATION_SIZE < 1 or IMAGE_SIZE < 32 or BATCH_SIZE < 1 or EPOCHS < 1:
        raise ValueError("Check validation size, image size, batch size, and epochs.")
    if LEARNING_RATE <= 0 or FINE_TUNE_LEARNING_RATE <= 0 or FINE_TUNE_LAYERS < 0:
        raise ValueError("Learning rates must be positive and FINE_TUNE_LAYERS cannot be negative.")

    paths, labels = (collect_image_samples(DATASET_PATH) if DATA_MODE == "image"
                     else collect_nifti_samples(DATASET_PATH, Path(LABELS_CSV)))
    class_names = validate_samples(paths, labels, VALIDATION_SIZE)
    class_to_index = {name: index for index, name in enumerate(class_names)}
    encoded_labels = [class_to_index[label] for label in labels]

    np.random.seed(SEED)
    tf.keras.utils.set_random_seed(SEED)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "plots").mkdir(exist_ok=True)
    patient_ids = read_patient_ids(Path(PATIENT_CSV), paths, DATASET_PATH) if PATIENT_CSV else None

    if patient_ids is not None:
        from sklearn.model_selection import StratifiedGroupKFold
        group_count = len(set(patient_ids))
        if group_count < 2:
            raise ValueError("Patient-level splitting needs at least two distinct patient IDs.")
        splitter = StratifiedGroupKFold(n_splits=min(5, group_count), shuffle=True, random_state=SEED)
        valid_splits = (
            (train, validation) for train, validation in splitter.split(paths, encoded_labels, groups=patient_ids)
            if len({encoded_labels[i] for i in train}) == len(class_names)
            and len({encoded_labels[i] for i in validation}) == len(class_names)
        )
        try:
            train_indices, validation_indices = min(
                valid_splits, key=lambda split: abs(len(split[1]) / len(paths) - VALIDATION_SIZE)
            )
        except ValueError as exc:
            raise ValueError("Could not make a patient-level split containing every class; add patients or review labels.") from exc
        train_paths = [paths[i] for i in train_indices]
        validation_paths = [paths[i] for i in validation_indices]
        y_train = [encoded_labels[i] for i in train_indices]
        y_validation = [encoded_labels[i] for i in validation_indices]
    else:
        train_paths, validation_paths, y_train, y_validation = train_test_split(
            paths, encoded_labels, test_size=VALIDATION_SIZE, random_state=SEED, stratify=encoded_labels
        )

    loader = (lambda sample: load_image(sample, IMAGE_SIZE, tf) if DATA_MODE == "image"
              else load_nifti_slice(sample, IMAGE_SIZE, nib, cv2, np))
    try:
        x_train = np.stack([loader(sample) for sample in train_paths])
        x_validation = np.stack([loader(sample) for sample in validation_paths])
    except (OSError, ValueError) as exc:
        raise ValueError(f"Could not load or prepare an MRI sample: {exc}") from exc
    y_train, y_validation = np.asarray(y_train), np.asarray(y_validation)

    train_counts = Counter(y_train.tolist())
    class_weights = {index: len(y_train) / (len(class_names) * train_counts[index]) for index in range(len(class_names))}
    label_map = {"class_to_index": class_to_index, "data_mode": DATA_MODE, "image_size": IMAGE_SIZE}
    (OUTPUT_DIR / "label_map.json").write_text(json.dumps(label_map, indent=2), encoding="utf-8")

    tf.keras.backend.clear_session()
    model = build_model(IMAGE_SIZE, len(class_names), WEIGHTS, tf, LEARNING_RATE)
    best_model_path = OUTPUT_DIR / "best_model.keras"
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(best_model_path, monitor="val_loss", save_best_only=True),
    ]
    history = model.fit(
        x_train, y_train, validation_data=(x_validation, y_validation), epochs=EPOCHS,
        batch_size=BATCH_SIZE, callbacks=callbacks, class_weight=class_weights,
    )
    if FINE_TUNE_LAYERS:
        unfreeze_top_layers(model, FINE_TUNE_LAYERS)
        model.compile(optimizer=tf.keras.optimizers.Adam(FINE_TUNE_LEARNING_RATE),
                      loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        fine_tune_history = model.fit(
            x_train, y_train, validation_data=(x_validation, y_validation), epochs=max(1, EPOCHS // 2),
            batch_size=BATCH_SIZE, callbacks=callbacks, class_weight=class_weights,
        )
        for metric, values in fine_tune_history.history.items():
            history.history[metric] = history.history.get(metric, []) + values

    model = tf.keras.models.load_model(best_model_path)
    predictions = np.argmax(model.predict(x_validation, verbose=0), axis=1)
    report = classification_report(y_validation, predictions, target_names=class_names, zero_division=0, output_dict=True)
    evaluation = {
        "classes": class_names,
        "class_counts": dict(Counter(labels)),
        "train_class_weights": class_weights,
        "confusion_matrix": confusion_matrix(y_validation, predictions).tolist(),
        "classification_report": report,
        "validation_accuracy": float((predictions == y_validation).mean()),
        "validation_recall_by_class": {name: report[name]["recall"] for name in class_names},
        "split_unit": "patient" if patient_ids else "image",
    }
    (OUTPUT_DIR / "evaluation.json").write_text(json.dumps(evaluation, indent=2), encoding="utf-8")
    model.save(OUTPUT_DIR / "final_model.keras")

    figure, axes = plt.subplots(1, 2, figsize=(10, 4))
    for axis, metric in zip(axes, ("accuracy", "loss")):
        axis.plot(history.history[metric], label="train")
        axis.plot(history.history[f"val_{metric}"], label="validation")
        axis.set_title(metric.capitalize())
        axis.grid(True)
        axis.legend()
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / "plots" / "training_history.png", dpi=150)
    plt.show()
    plt.close(figure)
    print(f"Completed. Validation accuracy: {evaluation['validation_accuracy']:.3f}")
    return evaluation

# After reviewing the configuration above, run:
# results = train_classifier()


## Predict one ordinary image
This inference helper applies only to a model trained in `image` mode. NIfTI volume prediction needs modality-aware preprocessing.


In [ ]:
def predict_image(image_path, model_path=None, label_map_path=None):
    """Return class scores for one 2D image using artifacts from the same training run."""
    image_path = Path(image_path)
    model_path = Path(model_path or OUTPUT_DIR / "best_model.keras")
    label_map_path = Path(label_map_path or OUTPUT_DIR / "label_map.json")
    if not image_path.is_file() or not model_path.is_file() or not label_map_path.is_file():
        raise FileNotFoundError("Image, model, and label-map files must all exist.")
    metadata = json.loads(label_map_path.read_text(encoding="utf-8"))
    if metadata.get("data_mode") != "image":
        raise ValueError("This model was trained in NIfTI mode; use a matching volume-inference pipeline.")
    image_size = int(metadata["image_size"])
    index_to_class = {int(index): name for name, index in metadata["class_to_index"].items()}
    if not index_to_class:
        raise ValueError("The label map has no class mapping.")
    model = tf.keras.models.load_model(model_path)
    scores = model.predict(np.expand_dims(load_image(image_path, image_size, tf), axis=0), verbose=0)[0]
    if len(scores) != len(index_to_class):
        raise ValueError("Model output count does not match the label map.")
    return {
        "prediction": index_to_class[int(np.argmax(scores))],
        "confidence": float(np.max(scores)),
        "scores": {index_to_class[index]: float(score) for index, score in enumerate(scores)},
    }

# print(predict_image("/content/drive/MyDrive/example.png"))
